# Framing COVID-19 Response and Vaccines Across Ideologies


This notebook analyzes ideological differences in COVID-19 and vaccine-related news headlines using natural language processing and machine-learning methods.

The project focuses on three questions:

1. How did liberal and conservative news outlets frame COVID-19 vaccination and pandemic response differently?
2. How did these framing patterns change from 2020 through 2022?
3. How did general vaccine-related media coverage change before versus after the COVID-19 pandemic?

The primary pandemic dataset contains more than 42,000 filtered headlines from six major news outlets. The analyses below include TF-IDF, logistic regression, Word2Vec embeddings, cosine similarity, and log-odds ratios.

> **Collaboration:** This project was completed with Molly Murphy for QTM 340 at Emory University. See the repository README for individual contribution details.

## 1. Setup and Imports

The original course notebook contained repeated imports, package-installation cells, and Google Drive paths. This portfolio version consolidates setup in one place and uses repository-relative file paths.

In [ ]:
from pathlib import Path
import math
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import gensim.downloader
from gensim.models import Word2Vec

# Download tokenizer resources if they are not already installed.
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

DATA_DIR = Path("../data")

## 2. Data Loading and Preparation

This notebook expects three processed files in the repository's `data/` folder:

- `filtered_covid_vaccine_headlines.csv` — COVID-19/pandemic headlines from 2020–2022
- `filtered_vaccine_headlines_2019.csv` — vaccine-related headlines from 2019
- `filtered_vaccine_headlines_2024.csv` — vaccine-related headlines from 2024

The raw source datasets are not included because of their size. See `data/README.md` in the repository for source information and filtering details.

In [ ]:
pandemic = pd.read_csv(DATA_DIR / "filtered_covid_vaccine_headlines.csv")
vax_2019 = pd.read_csv(DATA_DIR / "filtered_vaccine_headlines_2019.csv")
vax_2024 = pd.read_csv(DATA_DIR / "filtered_vaccine_headlines_2024.csv")

for df in [pandemic, vax_2019, vax_2024]:
    df["Headline"] = df["Headline"].astype(str).str.strip()
    df.dropna(subset=["Headline", "Ideology"], inplace=True)

pandemic["Date"] = pd.to_datetime(pandemic["Date"], errors="coerce")
pandemic["Year"] = pandemic["Date"].dt.year

print(f"Pandemic headlines: {len(pandemic):,}")
print(pandemic["Ideology"].value_counts())

In [ ]:
def clean_headlines(df, max_words=80):
    """Remove duplicate, empty, and unusually long headline records."""
    cleaned = df.copy()
    cleaned["word_count"] = cleaned["Headline"].str.split().str.len()
    cleaned = cleaned[
        cleaned["Headline"].notna()
        & cleaned["word_count"].between(1, max_words)
    ]
    cleaned = cleaned.drop_duplicates(subset=["Headline"])
    return cleaned.drop(columns="word_count")

pandemic = clean_headlines(pandemic)
vax_2019 = clean_headlines(vax_2019)
vax_2024 = clean_headlines(vax_2024)

## 3. Exploratory Data Analysis

Before modeling, we summarize the distribution of headlines across ideology, year, and publication. These checks help identify class imbalance and differences in source volume.

In [ ]:
summary = (
    pandemic.groupby(["Year", "Ideology"])
    .size()
    .unstack(fill_value=0)
)
summary

In [ ]:
publication_counts = (
    pandemic.groupby(["Publication", "Ideology"])
    .size()
    .sort_values(ascending=False)
)
publication_counts

## 4. Ideological Differences in COVID-19 Coverage

### 4.1 TF-IDF and Logistic Regression

Headlines are represented using TF-IDF unigram and bigram features. A logistic regression classifier is then trained to distinguish liberal from conservative outlets. Model coefficients are used to identify the n-grams most predictive of each ideological group.

In [ ]:
X_text = pandemic["Headline"]
y = pandemic["Ideology"]

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3
)
X = vectorizer.fit_transform(X_text)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

classifier = LogisticRegression(
    max_iter=300,
    class_weight="balanced"
)
classifier.fit(X_train, y_train)

pred = classifier.predict(X_test)

print(classification_report(y_test, pred))

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = classifier.coef_[0]

coef_series = pd.Series(coefficients, index=feature_names)
top_liberal = coef_series.sort_values(ascending=False).head(20)
top_conservative = coef_series.sort_values().head(20)

print("Top predictive liberal n-grams:")
display(top_liberal.to_frame("coefficient"))

print("\nTop predictive conservative n-grams:")
display(top_conservative.to_frame("coefficient"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8))

axes[0].barh(top_liberal.index, top_liberal.values)
axes[0].invert_yaxis()
axes[0].set_title("Top Predictive Liberal N-grams")
axes[0].set_xlabel("Logistic Regression Coefficient")

axes[1].barh(top_conservative.index, top_conservative.values)
axes[1].invert_yaxis()
axes[1].set_title("Top Predictive Conservative N-grams")
axes[1].set_xlabel("Logistic Regression Coefficient")

plt.suptitle("Most Predictive N-grams by Ideology")
plt.tight_layout()
plt.show()

**Interpretation:** In the final analysis, liberal outlets tended to emphasize scientific and situational updates, while conservative outlets more often emphasized political figures and government authority.

### 4.2 Word2Vec Semantic Framing

To examine how the same concepts appeared in different semantic contexts, separate Word2Vec models are trained for liberal and conservative headlines. The models are initialized with pretrained GloVe vectors and then fine-tuned on each corpus.

In [ ]:
STOP_WORDS = set(stopwords.words("english"))
PUNCT = set(string.punctuation)

def tokenize_headline(text):
    if not isinstance(text, str):
        return []
    tokens = word_tokenize(text.lower())
    return [
        token for token in tokens
        if token not in STOP_WORDS
        and token not in PUNCT
        and token.replace("-", "").isalpha()
    ]

pandemic["tokens"] = pandemic["Headline"].apply(tokenize_headline)

liberal = pandemic[pandemic["Ideology"] == "Liberal"].copy()
conservative = pandemic[pandemic["Ideology"] == "Conservative"].copy()

In [ ]:
# This download occurs once and requires an internet connection.
pretrained = gensim.downloader.load("glove-wiki-gigaword-100")

def finetune_model(pretrained_model, corpus, epochs=3):
    model = Word2Vec(
        vector_size=pretrained_model.vector_size,
        window=5,
        min_count=1,
        workers=4
    )
    model.build_vocab(corpus)

    for word in model.wv.key_to_index:
        if word in pretrained_model.key_to_index:
            model.wv[word] = pretrained_model[word]

    model.train(
        corpus,
        total_examples=model.corpus_count,
        epochs=epochs
    )
    return model

lib_model = finetune_model(pretrained, liberal["tokens"].tolist())
cons_model = finetune_model(pretrained, conservative["tokens"].tolist())

In [ ]:
def clean_neighbors(neighbors, min_len=3, top_n=10):
    cleaned = []
    for word, similarity in neighbors:
        display_word = word.strip("'").strip('"')
        if len(display_word) < min_len:
            continue
        if not display_word.replace("-", "").isalpha():
            continue
        cleaned.append((display_word, similarity))
    return cleaned[:top_n]

target_word = "mandate"

lib_neighbors = clean_neighbors(lib_model.wv.most_similar(target_word, topn=25))
cons_neighbors = clean_neighbors(cons_model.wv.most_similar(target_word, topn=25))

pd.DataFrame({
    "Liberal neighbor": [w for w, _ in lib_neighbors],
    "Liberal similarity": [s for _, s in lib_neighbors],
    "Conservative neighbor": [w for w, _ in cons_neighbors],
    "Conservative similarity": [s for _, s in cons_neighbors],
})

**Interpretation:** The final report found that liberal coverage associated *mandate* with administrative and public-health language, while conservative coverage associated it more strongly with rights-oriented or oppositional language.

## 5. Changes in Framing Over Time (2020–2022)

### 5.1 Year-Specific Word2Vec Models

In [ ]:
year_models = {}

for year in [2020, 2021, 2022]:
    year_df = pandemic[pandemic["Year"] == year]

    lib_year = year_df[year_df["Ideology"] == "Liberal"]["tokens"].tolist()
    cons_year = year_df[year_df["Ideology"] == "Conservative"]["tokens"].tolist()

    year_models[(year, "Liberal")] = finetune_model(pretrained, lib_year)
    year_models[(year, "Conservative")] = finetune_model(pretrained, cons_year)

### 5.2 Semantic Similarity Over Time

The analysis tracks how relationships between key concepts changed over time. The final paper highlighted the pairs `mask`–`freedom` and `vaccine`–`freedom`.

In [ ]:
pairs = [("mask", "freedom"), ("vaccine", "freedom")]
semantic_results = []

for year in [2020, 2021, 2022]:
    for ideology in ["Liberal", "Conservative"]:
        model = year_models[(year, ideology)]
        for word1, word2 in pairs:
            if word1 in model.wv and word2 in model.wv:
                similarity = model.wv.similarity(word1, word2)
                semantic_results.append({
                    "Year": year,
                    "Ideology": ideology,
                    "Pair": f"{word1}–{word2}",
                    "Cosine Similarity": similarity
                })

semantic_results = pd.DataFrame(semantic_results)
semantic_results

In [ ]:
for pair in semantic_results["Pair"].unique():
    subset = semantic_results[semantic_results["Pair"] == pair]

    plt.figure(figsize=(7, 4))
    for ideology in ["Liberal", "Conservative"]:
        group = subset[subset["Ideology"] == ideology]
        plt.plot(
            group["Year"],
            group["Cosine Similarity"],
            marker="o",
            label=ideology
        )

    plt.title(f"Semantic Similarity Over Time: {pair}")
    plt.xlabel("Year")
    plt.ylabel("Cosine Similarity")
    plt.xticks([2020, 2021, 2022])
    plt.legend()
    plt.show()

### 5.3 Ideological Classification by Year

Separate TF-IDF/logistic regression classifiers are trained for each year. This measures how distinguishable liberal and conservative headline language became as the pandemic progressed.

In [ ]:
def run_yearly_classifier(df, year):
    year_df = df[df["Year"] == year].dropna(subset=["Headline"]).copy()

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=3
    )

    X = vectorizer.fit_transform(year_df["Headline"])
    y = year_df["Ideology"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=100,
        stratify=y
    )

    model = LogisticRegression(
        max_iter=300,
        class_weight="balanced"
    )
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    return accuracy_score(y_test, predictions)

yearly_accuracy = pd.DataFrame({
    "Year": [2020, 2021, 2022],
    "Accuracy": [
        run_yearly_classifier(pandemic, 2020),
        run_yearly_classifier(pandemic, 2021),
        run_yearly_classifier(pandemic, 2022),
    ]
})

yearly_accuracy

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(yearly_accuracy["Year"], yearly_accuracy["Accuracy"], marker="o")
plt.title("Ideological Classifier Accuracy by Year")
plt.xlabel("Year")
plt.ylabel("Accuracy")
plt.xticks([2020, 2021, 2022])
plt.ylim(0.70, 0.90)
plt.show()

**Reported result:** The final paper reports classifier accuracies of approximately **0.78 (2020), 0.81 (2021), and 0.82 (2022)**, suggesting that ideological differences in headline language became more pronounced over time.

## 6. Vaccine Coverage Before vs. After COVID-19

### 6.1 Prepare 2019 and 2024 Vaccine Data

In [ ]:
for df, year in [(vax_2019, 2019), (vax_2024, 2024)]:
    df["Year"] = year
    df["text"] = df["Headline"].astype(str)
    df["tokens"] = df["Headline"].apply(tokenize_headline)

pre_post = pd.concat([vax_2019, vax_2024], ignore_index=True)

pre_post.groupby(["Year", "Ideology"]).size().unstack(fill_value=0)

### 6.2 TF-IDF Term Comparison

TF-IDF is used separately for 2019 and 2024 to identify the terms most distinctive within vaccine-related coverage in each period.

In [ ]:
def top_tfidf_terms(text_series, n=30):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )
    matrix = vectorizer.fit_transform(text_series.astype(str))
    features = vectorizer.get_feature_names_out()
    mean_scores = np.asarray(matrix.mean(axis=0)).ravel()
    top_idx = mean_scores.argsort()[::-1][:n]
    return pd.DataFrame({
        "term": features[top_idx],
        "mean_tfidf": mean_scores[top_idx]
    })

top_2019 = top_tfidf_terms(vax_2019["Headline"])
top_2024 = top_tfidf_terms(vax_2024["Headline"])

display(top_2019.head(20))
display(top_2024.head(20))

**Interpretation:** The final analysis found that 2019 coverage was dominated by measles outbreaks and anti-vaccination discourse, while 2024 coverage focused more on COVID-19, flu, polio, mpox, and other infectious-disease threats.

### 6.3 Log-Odds Analysis

Log-odds ratios identify words disproportionately used by conservative versus liberal outlets in each period.

In [ ]:
def smooth_prob(vocab, word, total_tokens, alpha=1):
    vocabulary_size = len(vocab)
    count = vocab.get(word, 0)
    return (count + alpha) / (total_tokens + vocabulary_size * alpha)

def build_vocab(df):
    vocab = {}
    for tokens in df["tokens"]:
        for token in tokens:
            if token.isalpha() and len(token) > 1:
                vocab[token] = vocab.get(token, 0) + 1
    return vocab

def log_odds_ratio(words, vocab1, vocab2):
    total1 = sum(vocab1.values())
    total2 = sum(vocab2.values())
    results = []

    for word in words:
        p1 = smooth_prob(vocab1, word, total1)
        p2 = smooth_prob(vocab2, word, total2)

        log_odds1 = math.log(p1 / (1 - p1 + 1e-12))
        log_odds2 = math.log(p2 / (1 - p2 + 1e-12))
        results.append((word, log_odds1 - log_odds2))

    return results

def ideology_log_odds(df):
    cons = df[df["Ideology"] == "Conservative"]
    lib = df[df["Ideology"] == "Liberal"]

    cons_vocab = build_vocab(cons)
    lib_vocab = build_vocab(lib)
    words = sorted(set(cons_vocab) | set(lib_vocab))

    scores = log_odds_ratio(words, cons_vocab, lib_vocab)
    return sorted(scores, key=lambda x: x[1], reverse=True)

log_odds_2019 = ideology_log_odds(vax_2019)
log_odds_2024 = ideology_log_odds(vax_2024)

In [ ]:
def plot_log_odds(scores, year, top_n=15):
    conservative_terms = scores[:top_n]
    liberal_terms = scores[-top_n:]

    cons_words = [word for word, _ in conservative_terms]
    cons_scores = [score for _, score in conservative_terms]

    lib_words = [word for word, _ in liberal_terms]
    lib_scores = [score for _, score in liberal_terms]

    plt.figure(figsize=(9, 7))
    plt.barh(cons_words, cons_scores, label="Conservative")
    plt.barh(lib_words, lib_scores, label="Liberal")
    plt.axvline(0, linewidth=1)
    plt.title(f"Exclusive & Overrepresented Words in Vaccine Headlines — {year}")
    plt.xlabel("Log-Odds Strength")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_log_odds(log_odds_2019, 2019)
plot_log_odds(log_odds_2024, 2024)

**Interpretation:** The final report found that pre-pandemic liberal coverage emphasized health measures and community impacts, while conservative coverage emphasized more personal or emotional narratives. In 2024, liberal outlets emphasized global outbreaks and scientific reporting, while conservative outlets showed more vaccine-skeptical language.

## 7. Key Findings

- Liberal and conservative outlets showed distinct linguistic framing of COVID-19 and public-health policy.
- Word2Vec models revealed different semantic contexts for policy terms such as *mandate*.
- Ideological classification became more accurate from 2020 through 2022, consistent with increasingly distinguishable headline language.
- Vaccine-related media coverage shifted substantially between 2019 and 2024, from measles and anti-vaccination discourse toward a broader set of infectious-disease threats.

## 8. Limitations

- The analysis uses headlines rather than full articles, limiting contextual information.
- The selected publications represent only a subset of the broader media landscape.
- Keyword filtering may exclude relevant headlines.
- Differences in dataset collection between 2019 and 2024 should be considered when interpreting pre/post comparisons.